# Description: Final steps of the tedana pipeline for the "evaluation" dataset

This notebook implements the extra steps (beyond ```afni_proc```) needed to complete the per-echo **TEDANA/fastica pipeline** which includes the following operations:

* Corrects the space in relevant tedana outcomes with ```3drefit```
* Takes the indivudally denoised echoes from the tedana folder and names then pb06 on the main pre-processed folder
* Computes the GS for the OC dataset
* Regresses additional items beyond those of the tedana process
* Extracts ROI timeseries
* Computed within-echo FC
* Computes TSNR map


In [1]:
import pandas as pd
import numpy as np
import os.path as osp
import datetime
import os
from tqdm import tqdm
from utils.basics import PRCS_DATA_DIR, ATLASES_DIR, ATLAS_NAME, PRJ_DIR, CODE_DIR
from utils.basics import get_dataset_index

Obtain current user name. We use this to compose SWARM file and log folder paths

In [2]:
import getpass
username = getpass.getuser()
print(username)

javiergc


Select dataset and atlas

In [3]:
DATASET    = 'evaluation'
ATLAS_NAME = f'Power264-{DATASET}'
ATLAS_DIR  = osp.join(ATLASES_DIR,ATLAS_NAME)

# 1. Load Dataset Information

In [4]:
ds_index = get_dataset_index(DATASET)
ses_list = list(ds_index.get_level_values('Session').unique())
sbj_list = list(ds_index.get_level_values('Subject').unique())

++ Number of scans    = 439
++ Number of subjects = 221


# 2. Create Swarm Script to Extract ROI TS from fully denoised data

First, we create the path to the SWARM file with all the needed batch jobs

In [5]:
script_path = osp.join(PRJ_DIR,f'swarm.{username}',f'N03b_Tedana_pipelines_and_ROIextract.{ATLAS_NAME}.SWARM.sh')
print(script_path)

/data/SFIMJGC_HCP7T/BCBL2024/swarm.javiergc/N03b_Tedana_pipelines_and_ROIextract.Power264-evaluation.SWARM.sh


Second, we create the folders for the log files generated by the different batch jobs 

In [6]:
log_path = osp.join(PRJ_DIR,f'logs.{username}',f'N03b_Tedana_pipeline_and_ROIextract.{ATLAS_NAME}.log')
if not osp.exists(log_path):
    os.makedirs(log_path)
print(log_path)

/data/SFIMJGC_HCP7T/BCBL2024/logs.javiergc/N03b_Tedana_pipeline_and_ROIextract.Power264-evaluation.log


Third, we generate the SWARM file. This file will contain one line per scan that calls the ```S09_Tedana_pipelines_and_ROIextract.sh``` script.

> **NOTE:** the TEDANA_TYPE variable would allow to work with outputs of tedana run in different ways (e.g., robustica, component selection with MLD, etc.). In this project we only work with the default case (fastica).

In [7]:
atlas_path  = f'{ATLASES_DIR}/{ATLAS_NAME}/{ATLAS_NAME}.nii.gz' 
with open(script_path, 'w') as the_file:
    the_file.write('# Script Creation Date: %s\n' % str(datetime.date.today()))
    the_file.write(f'# swarm -f {script_path} -g 16 -t 8 -b 2 --time 02:00:00 --logdir {log_path} --partition quick,norm --module afni\n')
    the_file.write('\n')
    for sbj,ses in list(ds_index):
        for NORDIC in ['off','on']:
            the_file.write(f'export SBJ={sbj} SES={ses} NORDIC={NORDIC} TEDANA_TYPE=fastica ATLAS_NAME={ATLAS_NAME} ATLAS_PATH={atlas_path} ATLASES_DIR={ATLASES_DIR}; sh  {CODE_DIR}/bash/S10_Tedana_pipelines_and_ROIextract.sh \n')
the_file.close()     

print('Swarm script saved to %s' % script_path)

Swarm script saved to /data/SFIMJGC_HCP7T/BCBL2024/swarm.javiergc/N03b_Tedana_pipelines_and_ROIextract.Power264-evaluation.SWARM.sh


To submit the SWARM file, login into Biowulf and run the following commands on a terminal:

```bash
cd /data/SFIMJGC_HCP7T/BCBL2024/swarm.javiergc

swarm -f /data/SFIMJGC_HCP7T/BCBL2024/swarm.javiergc/N03b_Tedana_pipeline_and_ROIextract.Power264-evaluation.SWARM.sh \
      -g 16 -t 8 -b 2 --time 02:00:00 \
      --logdir /data/SFIMJGC_HCP7T/BCBL2024/logs.javiergc/N03b_Tedana_pipeline_and_ROIextract.Power264-evaluation.log \
      --partition quick,norm --module afni
```

# 3. Check all expected datasets were processed

In [8]:
%%time
num_missing_files = 0
for sbj,ses in list(ds_index):
    for scenario in ['ALL_Tedana-fastica']:
        for NORDIC in ['off','on']:
            for e in ['e01','e02','e03']:
                netcc_path = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-{NORDIC}',f'errts.{sbj}.r01.{e}.volreg.spc.tproject_{scenario}.{ATLAS_NAME}_000.netts')
                if not osp.exists(netcc_path):
                    print('++ WARNING: %s is missing'% netcc_path)
                    num_missing_files +=1
print('+ Number of incomplete jobs: %d' % num_missing_files)

+ Number of incomplete jobs: 0
CPU times: user 45.6 ms, sys: 157 ms, total: 202 ms
Wall time: 3.19 s


***

# 4. Create a small dashboard to explore FC matrices for different pipelines

The remaining of the code is for exploratory purposes. Nothing beyond this point is needed in the rest of the analyses.

In [9]:
import xarray as xr
import panel as pn
from sfim_lib.io.afni import load_netcc
from sfim_lib.plotting.fc_matrices import hvplot_fc

/data/SFIMJGC_HCP7T/Apps/envs/generic_2025a/lib/python3.12/site-packages/nxviz/__init__.py:18: UserWarning: 
nxviz has a new API! Version 0.7.4 onwards, the old class-based API is being
deprecated in favour of a new API focused on advancing a grammar of network
graphics. If your plotting code depends on the old API, please consider
pinning nxviz at version 0.7.4, as the new API will break your old code.

To check out the new API, please head over to the docs at
https://ericmjl.github.io/nxviz/ to learn more. We hope you enjoy using it!

(This deprecation message will go away in version 1.0.)

  warnings.warn(


Load information about the ROIs in the ATLAS. This is needed for plotting purposes.

In [10]:
roi_info_path = osp.join(ATLAS_DIR,f'{ATLAS_NAME}.roi_info.csv')
roi_info_df   = pd.read_csv(roi_info_path)

Create a colormap that will assign a different color to each network

In [11]:
power264_nw_cmap = {nw:roi_info_df.set_index('Network').loc[nw]['RGB'].values[0] for nw in list(roi_info_df['Network'].unique())}

Create an xr.DataArray that will hold the within echo FC matrices for all scans in all pipelines.

In [12]:
# NORDIC off scenarios
scenarios = [('Basic','ALL','off','e'+str(i+1).zfill(2)) for i in np.arange(3)] + \
            [('GS','ALL','off','e'+str(i+1).zfill(2)) for i in np.arange(3)] + \
            [('Tedana-fastica','ALL','off','e'+str(i+1).zfill(2)) for i in np.arange(3)]
# NORDIC on scenarios
scenarios = scenarios + \
            [('Basic','ALL','on','e'+str(i+1).zfill(2)) for i in np.arange(3)] + \
            [('GS','ALL','on','e'+str(i+1).zfill(2)) for i in np.arange(3)] + \
            [('Tedana-fastica','ALL','on','e'+str(i+1).zfill(2)) for i in np.arange(3)]

fcs = {scenario:xr.DataArray(dims=['scan','roi_x','roi_y'], coords={'scan':['.'.join([sbj,ses]) for sbj,ses in ds_index],
                                                                           'roi_x':list(roi_info_df['ROI_Name']),
                                                                           'roi_y':list(roi_info_df['ROI_Name'])}) for scenario in scenarios}

Load all the within-echo FC matrices

In [13]:
for sbj,ses in tqdm(list(ds_index)):
    for pipeline,censor_mode,NORDIC,te in scenarios:
        netcc_path = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-{NORDIC}',f'errts.{sbj}.r01.{te}.volreg.spc.tproject_{censor_mode}_{pipeline}.{ATLAS_NAME}_000.netcc')
        netcc = load_netcc(netcc_path)
        fcs[(pipeline,censor_mode,NORDIC,te)].loc['.'.join([sbj,ses]),:,:] = netcc.values

100%|██████████| 439/439 [01:56<00:00,  3.76it/s]


Load Motion Timeseries

In [14]:
FINAL_N_ACQS = 201
mot_df = pd.DataFrame(index=np.arange(FINAL_N_ACQS),columns=list(ds_index))
for sbj,ses in tqdm(list(ds_index)):
    mot_path = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off',f'motion_{sbj}_enorm.1D')
    mot_df[(sbj,ses)] = np.loadtxt(mot_path)

  0%|          | 0/439 [00:00<?, ?it/s]

100%|██████████| 439/439 [00:00<00:00, 673.96it/s]


Create all the necessary dashboard elements

In [15]:
scan_select = pn.widgets.Select(name='scan', options=list(ds_index))
@pn.depends(scan_select)
def plot_fc(scan):
    sbj = scan[0]
    ses = scan[1]
    scan_index = '.'.join([sbj,ses])
    plots = pn.layout.GridBox(ncols=3)
    for scenario in scenarios:
        aux = fcs[scenario].sel(scan=scan_index).values
        aux = pd.DataFrame(aux,index=roi_info_df.set_index(['ROI_Name','ROI_ID','Hemisphere','Network','RGB']).index,
                               columns=roi_info_df.set_index(['ROI_Name','ROI_ID','Hemisphere','Network','RGB']).index)
        aux_plot = hvplot_fc(aux,major_label_overrides='regular_grid',cmap='RdBu_r', by='Network', add_labels=False, colorbar_position='left', net_cmap=power264_nw_cmap, cbar_title='%s-%s-%s-%s' % scenario)
        plots.append(aux_plot)
    return pn.Row(plots)

In [16]:
@pn.depends(scan_select)
def plot_mot(scan):
    aux_df = pd.DataFrame(mot_df[scan].values,columns=['Motion [enorm]'])
    aux_df.index.name = 'TR'
    aux_df.name = 'Motion'
    return aux_df.hvplot()

Start the Dashboard

In [ ]:
dashboard = pn.Row(scan_select,pn.Column(plot_fc,plot_mot)).show()

Launching server at http://localhost:38413


In [31]:
dashboard.stop()